### 환경설정

`.env`에서 IMAP 접속 정보(`IMAP_HOST`, `EMAIL_ADDR`, `EMAIL_PASSWORD`)를 불러오고, 결과를 저장할 `output/` 폴더를 준비합니다.

In [1]:
import os
from pathlib import Path

import imaplib
from dotenv import load_dotenv

load_dotenv()  # 프로젝트 루트(work-code-archive-organized)의 .env 파일을 자동으로 찾아 로드

IMAP_SERVER = os.environ["IMAP_HOST"]
IMAP_PORT = int(os.environ.get("IMAP_PORT", "993"))
EMAIL = os.environ["EMAIL_ADDR"]
PASSWORD = os.environ["EMAIL_PASSWORD"]

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

IMAP 서버에 로그인해 접속 정보가 유효한지 확인합니다.

In [2]:
imap = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
imap.login(EMAIL, PASSWORD)

print("✅ IMAP 로그인 성공")

✅ IMAP 로그인 성공


# 조아기프트

#### 메일 가져오기

수집 대상 메일함 목록입니다. 조아기프트 발주서가 쌓인 연도별 메일함 6개(2018~2023)를 지정합니다.

In [3]:
import email
from email.header import decode_header

JOA_FOLDERS = [
    "&yHDFRK4w1QTSuA-_230217~",
    "&yHDFRK4w1QTSuA-_210706~",
    "&yHDFRK4w1QTSuA-_200519~",
    "&yHDFRK4w1QTSuA-_190711~",
    "&yHDFRK4w1QTSuA-_181002~",
    "&yHDFRK4w1QTSuA-_180111~",
]

### 1차 메일 수집 — 원본 bytes 덤프 + HTML 본문 디코딩

메일 원문을 `본문_bytes`로 그대로 보관한 뒤(`01-01_raw_mail_dump.pkl`), HTML 파트는 `td.Hidden_tex1` 선택자로 디코딩합니다(`01-02_html_decoded.xlsx`).

메일 제목(Subject) 헤더를 디코딩하는 함수입니다. 인코딩 정보가 없거나 깨진 경우 utf-8/cp949/euc-kr/iso-2022-kr을 순서대로 시도합니다.

In [4]:
ENCODINGS = ['utf-8', 'cp949', 'euc-kr', 'iso-2022-kr']

def decode_str_safe(raw_str):
    if not raw_str:
        return None
    decoded_parts = decode_header(raw_str)
    result_parts = []
    for part, enc in decoded_parts:
        if isinstance(part, bytes):
            # 우선 enc 사용, 없으면 후보 인코딩 시도
            tried = False
            if enc:
                try:
                    part = part.decode(enc)
                    tried = True
                except (UnicodeDecodeError, LookupError):
                    pass
            if not tried:
                for e in ENCODINGS:
                    try:
                        part = part.decode(e)
                        tried = True
                        break
                    except (UnicodeDecodeError, LookupError):
                        continue
            if not tried:
                return None  # 모든 시도 실패하면 None
        result_parts.append(part)
    return ''.join(result_parts)

메일함을 순회하며 각 메일의 제목/본문(bytes 원본)을 그대로 수집합니다. 본문은 디코딩하지 않고 bytes로 보관 → `01-01_raw_mail_dump.pkl`

In [5]:
import imaplib
import email
import pandas as pd
from email.header import decode_header

imap = imaplib.IMAP4_SSL(IMAP_SERVER)
imap.login(EMAIL, PASSWORD)

parsed_rows = []

for MAILBOX in JOA_FOLDERS:
    imap.select(MAILBOX)

    # 🔑 UID 기반 검색
    status, messages = imap.uid('search', None, 'ALL')
    uid_list = messages[0].split()

    for uid in uid_list:
        # 🔑 UID 기반 fetch
        res, msg_data = imap.uid('fetch', uid, '(RFC822)')
        if res != 'OK':
            continue

        msg = email.message_from_bytes(msg_data[0][1])

        # -----------------------
        # 고유 키 생성
        # -----------------------
        unique_key = f"{MAILBOX}|{uid.decode()}"

        # 제목
        raw_subject = msg.get("Subject")
        subject = decode_str_safe(raw_subject)

        body_bytes = None
        body_type = None

        if msg.is_multipart():
            for part in msg.walk():
                content_type = part.get_content_type()
                content_dispo = str(part.get("Content-Disposition"))

                if "attachment" in content_dispo:
                    continue

                if content_type in ("text/html", "text/plain"):
                    body_bytes = part.get_payload(decode=True)
                    body_type = content_type
                    if body_bytes:
                        break
        else:
            body_bytes = msg.get_payload(decode=True)
            body_type = msg.get_content_type()

        parsed_rows.append({
            "메일키": unique_key,       # ✅ 핵심
            "UID": uid.decode(),
            "메일제목": subject,
            "본문_bytes": body_bytes,   # ❗ decode 안 함 (좋은 판단)
            "본문_type": body_type,
            "메일폴더": MAILBOX
        })

imap.logout()

df_raw_dump = pd.DataFrame(parsed_rows)
df_raw_dump.to_pickle(OUTPUT_DIR / "01-01_raw_mail_dump.pkl")
print(f"총 {len(parsed_rows)}건 저장 완료")


총 2326건 저장 완료


1차 수집 결과의 본문 bytes를 디코딩합니다. HTML이면 `td.Hidden_tex1` 선택자로 발주서 표 부분만 추출하고, 그 외(plain)는 euc-kr로 디코딩 → `01-02_html_decoded.xlsx`

In [6]:
from bs4 import BeautifulSoup

order_df = df_raw_dump.copy()
from bs4 import BeautifulSoup
import chardet

def decode_html_bytes(body_bytes):
    # 1. charset 감지
    result = chardet.detect(body_bytes)
    encoding = result['encoding'] if result['encoding'] else 'utf-8'
    
    # 2. bytes -> str
    try:
        html_text = body_bytes.decode(encoding, errors='ignore')
    except:
        html_text = body_bytes.decode('utf-8', errors='ignore')
    
    # 3. BeautifulSoup으로 HTML 파싱
    soup = BeautifulSoup(html_text, "html.parser")
    
    # 4. 발주서 테이블 추출 (td.Hidden_tex1)
    tds = soup.select("td.Hidden_tex1")
    extracted = [td.get_text(strip=True) for td in tds if td.get_text(strip=True)]
    return "\n".join(extracted)

order_df["clean_text"] = order_df.apply(
    lambda row: decode_html_bytes(row["본문_bytes"]) 
                if row["본문_type"].lower() == "text/html" 
                else row["본문_bytes"].decode("euc-kr", errors="ignore").strip(),
    axis=1
)

order_df.to_excel(OUTPUT_DIR / "01-02_html_decoded.xlsx", index=False)

HTML 본문을 사람이 읽기 좋은 텍스트로 정제하는 함수입니다(`clean_body`). 태그 제거, `&nbsp;` 처리, 연속 줄바꿈/공백 축소.

In [7]:
from bs4 import BeautifulSoup
import pandas as pd
import re

def clean_body(raw_body):
    """
    HTML 본문 정리
    - BeautifulSoup로 HTML 태그 제거
    - &nbsp; 등 공백 제거
    - 연속 줄바꿈 1개로
    - 연속 공백 1개로
    """
    if not raw_body or pd.isna(raw_body):
        return ""  # NaN이나 None이면 빈 문자열
    
    soup = BeautifulSoup(raw_body, "html.parser")
    text = soup.get_text(separator="\n")  # <br>, <p> 등을 줄바꿈으로

    # 공백/줄바꿈 정리
    text = text.replace('\xa0', ' ')               # &nbsp;
    text = re.sub(r'\r\n', '\n', text)            # 윈도우 개행 통일
    text = re.sub(r'\n\s*\n+', '\n', text)        # 연속 줄바꿈 -> 1개
    text = re.sub(r'[ ]{2,}', ' ', text)          # 연속 공백 -> 1개
    return text.strip()


### 2차 메일 수집 — 전체 재조회 + clean_body 정제

1차 수집과 별개로 전체 메일함을 다시 조회하면서 `clean_body()`로 HTML/plain 본문을 정제합니다(`01-03_plain_html_reparsed.xlsx`). 1차 결과의 plain 파트를 이 결과로 보완하는 데 사용됩니다.

In [8]:
imap = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
imap.login(EMAIL, PASSWORD)

parsed_rows = []

for MAILBOX in JOA_FOLDERS:
    imap.select(MAILBOX)

    # 🔑 UID 기반 검색
    status, messages = imap.uid('search', None, 'ALL')
    uid_list = messages[0].split()
    
    for uid in uid_list:
        # UID로 fetch
        status, msg_data = imap.uid('fetch', uid, "(RFC822)")
        if status != "OK":
            continue

        raw_email = msg_data[0][1]
        msg = email.message_from_bytes(raw_email)

        unique_key = f"{MAILBOX}|{uid.decode()}"
        
        # 본문 추출 (HTML 우선)
        body = ""
        if msg.is_multipart():
            for part in msg.walk():
                content_type = part.get_content_type()
                content_dispo = str(part.get("Content-Disposition"))

                if "attachment" in content_dispo:
                    continue

                if content_type == "text/html":   # <- 여기 변경
                    body = part.get_payload(decode=True).decode('utf-8', errors='ignore')
                    break

        else:
            body = msg.get_payload(decode=True).decode('utf-8', errors='ignore')
            content_type = msg.get_content_type()

        soup = BeautifulSoup(body, "html.parser")
        text = soup.get_text(separator="\n")

        clean_text = clean_body(text)

        parsed_rows.append({
            "메일키": unique_key,
            "UID": uid.decode(),
            "메일폴더": MAILBOX,
            "본문_type": content_type,
            "raw_body": text,
            "clean_body": clean_text
        })

imap.logout()

('BYE', [b'IMAP4rev1 Server logging out'])

In [9]:
df_reparsed = pd.DataFrame(parsed_rows)

df_reparsed.to_excel(OUTPUT_DIR / "01-03_plain_html_reparsed.xlsx", index=False)
print(f"총 {len(parsed_rows)}건 저장 완료")

총 2326건 저장 완료


1차 수집(`01-02`)과 2차 수집(`01-03`)을 합칩니다. `본문_type`과 무관하게 2차 재조회에서 찾은 메일은 전부 `clean_body`를 채웁니다 → `01-04_html_decoded_plain_merged.xlsx`

**수정(2026-07-08)**: 원래는 `본문_type == "text/plain"`인 메일만 채웠는데, HTML 메일의 1차 추출(`td.Hidden_tex1` 선택자)이 "품 명"/"옵 션" 같은 라벨 셀을 통째로 빼먹는 문제가 발견되어, HTML 메일도 라벨이 살아있는 2차 결과(`clean_body`)를 쓰도록 조건을 없앴습니다.

In [ ]:
import pandas as pd

df_main = pd.read_excel(OUTPUT_DIR / "01-02_html_decoded.xlsx")
df_clean = pd.read_excel(OUTPUT_DIR / "01-03_plain_html_reparsed.xlsx")


clean_body_map = dict(
    zip(df_clean["메일키"], df_clean["clean_body"])
)

# 교체 대상 조건 (본문_type 상관없이 2차 재조회 결과가 있으면 전부 채움 —
# text/html 메일도 여기서 clean_body를 받아야 다음 단계에서 라벨 있는 본문을 쓸 수 있음)
mask = df_main["메일키"].isin(clean_body_map)

# clean_body 채우기
df_main.loc[mask, "clean_body"] = (
    df_main.loc[mask, "메일키"]
    .map(clean_body_map)
)

# 새로운 파일로 저장
output_file = OUTPUT_DIR / "01-04_html_decoded_plain_merged.xlsx"
df_main.to_excel(output_file, index=False)

print(f"✅ 교체 완료: {mask.sum()}건")
print(f"📁 새 파일 저장됨: {output_file}")

최종 `본문` 컬럼을 확정하는 규칙입니다: text/plain인데 수동보정 대상(`01-05`)에 있는 메일만 `clean_text`, 나머지(HTML 메일 포함)는 전부 `clean_body`를 사용 → `01-06_body_finalized.xlsx`

**수정(2026-07-08)**: HTML 메일도 `clean_text`(라벨 없음) 대신 `clean_body`(라벨 있음)를 쓰도록 변경.

In [ ]:
import pandas as pd

df_main = pd.read_excel(OUTPUT_DIR / "01-04_html_decoded_plain_merged.xlsx")
df_clean_text = pd.read_excel(OUTPUT_DIR / "01-05_manual_clean_text_override.xlsx")  # 메일키만 있음

# text/plain에서 clean_text를 쓸 메일키 set
plain_use_clean_text_keys = set(df_clean_text["메일키"])

# 본문 컬럼 생성
def select_body(row):
    if row["본문_type"] == "text/html":
        # HTML이면 라벨이 살아있는 clean_body 사용 (clean_text는 td.Hidden_tex1 선택자라 라벨이 빠짐)
        return row["clean_body"]
    elif row["본문_type"] == "text/plain" and row["메일키"] in plain_use_clean_text_keys:
        # text/plain인데 메일키가 df_clean_text에 있으면 clean_text
        return row["clean_text"]
    else:
        # 나머지는 clean_body
        return row["clean_body"]

df_main["본문"] = df_main.apply(select_body, axis=1)

# 새로운 파일로 저장
output_file = OUTPUT_DIR / "01-06_body_finalized.xlsx"
df_main.to_excel(output_file, index=False)

print(f"✅ 본문 컬럼 생성 완료, 파일 저장됨: {output_file}")


현재까지 만들어진 컬럼 구성을 눈으로 확인하는 점검용 셀입니다.

In [12]:
df_main.columns

Index(['메일키', 'UID', '메일제목', '본문_bytes', '본문_type', '메일폴더', 'clean_text',
       'clean_body', '본문'],
      dtype='str')

검토하기 편하도록 핵심 컬럼(제목/본문/메일키/UID/메일폴더)만 추려 저장합니다 → `01-07_mail_subset_for_review.xlsx`

이 파일에서 본문이 비어있는 45건을 수동으로 채운 것이 다음 단계의 `01-08`입니다.

In [13]:
df_main[['메일제목', '본문','메일키', 'UID', '메일폴더']].to_excel(OUTPUT_DIR / '01-07_mail_subset_for_review.xlsx', index=False)

### 메일 본문 필드 파싱

**사전 준비 필요**: 이 아래 셀은 `01-08_manual_fixed_missing_bodies.xlsx`(본문 누락 45건을 수동으로 고친 파일)가 `output/`에 이미 있어야 실행됩니다.

바로 다음 셀에서 이 `01-08` 파일을 불러와 파싱을 시작합니다.

In [14]:
import pandas as pd

df_mail = pd.read_excel(OUTPUT_DIR / '01-08_manual_fixed_missing_bodies.xlsx')
df_mail['본문']

0       발 주 서 ( 주문번호 : 5439674 )\n발 주 서\n(주)명성 귀하 (수령자...
1       발 주 서 ( 주문번호 : 5439869 )\n발 주 서\n(주)명성 귀하 (수령자...
2       발 주 서 ( 주문번호 : 5440210 )\n발 주 서\n(주)명성 귀하 (수령자...
3       발 주 서 ( 주문번호 : 5442983 )\n샘플신청서\n(주)명성 귀하 (수령자...
4       발 주 서 ( 주문번호 : 5444212 )\n발 주 서\n(주)명성 귀하 (수령자...
                              ...                        
2321    에스모도 887 듀얼케이블 일체형 5000mAh 보조배터리    -183-\n27일...
2322    조아기프트 시안좀 확인해주세요~!\n\n\n\n \n\n\n\n담당자 강혜련\n\n...
2323    에스모도 887 듀얼케이블 일체형 5000mAh 보조배터리    -183-\n60E...
2324    케이블 일체형 보조배터리 5000mAh    -183-\n90EA\n￦7,260\n...
2325    에스모도 887 듀얼케이블 일체형 5000mAh 보조배터리    -183-\n100...
Name: 본문, Length: 2326, dtype: str

발주서 본문에서 품명/옵션/수량/단가/금액/합계/발송일 등 필드를 정규식으로 추출하는 함수(`parse_order`)와, 파싱 전 줄바꿈/공백을 정리하는 `clean_text` 함수를 정의합니다.

In [15]:
import re

def clean_text(text):
    """
    - 연속된 공백 제거
    - 줄바꿈(\n) 제거
    - 불필요한 특수문자 제거 (선택)
    """
    if not isinstance(text, str):
        return ""
    
    # 1. 줄바꿈 제거
    text = text.replace('\n', ' ')
    
    # 2. 여러 공백을 하나로
    text = re.sub(r'\s+', ' ', text)
    
    # 3. 불필요한 연속 점, 특수문자 정리 (선택)
    text = re.sub(r'\.{2,}', '.', text)
    
    # 4. 양쪽 공백 제거
    return text.strip()


def parse_order(body):
    row = {}

    # 1. 품명
    m = re.search(r"품\s*명\s*(.+)", body)
    row['품명'] = m.group(1).strip() if m else ""

    # 2. 옵션
    m = re.search(r"옵\s*션\s*(.+)", body)
    row['옵션'] = m.group(1).strip() if m else ""

    # 3. 수량
    m = re.search(r"수\s*량\s*([\d,]+)EA", body)
    row['수량'] = int(m.group(1).replace(',', '')) if m else 0

    # 4. 단가
    m = re.search(r"단\s*가\s*￦([\d,]+)", body)
    row['단가'] = int(m.group(1).replace(',', '')) if m else 0

    # 5. 금액
    m = re.search(r"금\s*액\s*￦([\d,]+)", body)
    row['금액'] = int(m.group(1).replace(',', '')) if m else 0

    # 6. 합계
    m = re.search(r"합\s*계\s*￦([\d,]+)", body)
    row['합계'] = int(m.group(1).replace(',', '')) if m else 0

    # 7. 인쇄방법
    m = re.search(r"수\s*량\s*[\d,]+EA\s*★(.+)", body)
    row['인쇄방법'] = m.group(1).strip() if m else ""

    # 8. 발송일
    m = re.search(r"발\s*송\s*일\s*([0-9]{4}년 [0-9]{1,2}월 [0-9]{1,2}일)", body)
    row['발송일'] = m.group(1).strip() if m else ""

    # 9. 발주일
    m = re.findall(r"상기와 같이 발주 합니다\.\s*\n([0-9]{4}년 [0-9]{1,2}월 [0-9]{1,2}일)", body)
    row['발주일'] = m[-1].strip() if m else ""

    # 10. 도착주소
    m = re.search(r"도착주소\s*\n\s*(.+?)(?:\n\s*\S)", body, re.DOTALL)
    row['도착주소'] = m.group(1).strip() if m else ""

    # 11. 수령자
    m = re.search(r"수령자[:\s]*([^\)]+)\)?", body)
    row['수령자'] = m.group(1).strip() if m else ""

    # 12. 수령자 연락처
    m = re.search(r"T[:\s]*([\d-]+)", body)
    row['수령자 연락처'] = m.group(1).strip() if m else ""

    # 13. 담당자 메일
    m = re.search(r"메일\s*[:]\s*([\w\.-]+@[\w\.-]+)", body)
    row['담당자_메일'] = m.group(1).strip() if m else ""

    # 14. 비고
    m = re.search(r"비\s*고\s*\n(.+?)(?:\n\s*\n|첨부파일)", body, re.DOTALL)
    row['비고'] = m.group(1).strip() if m else ""

    return row


위 `parse_order`로 1차 파싱을 실행합니다(아직 줄바꿈이 남아있는 원본 본문 기준) → 다음 셀에서 `01-09_parsed_fields_draft.xlsx`로 저장

In [16]:
parsed_df = df_mail['본문'].apply(lambda x: parse_order(x) if isinstance(x, str) else parse_order("")).apply(pd.Series)

df_mail[parsed_df.columns] = parsed_df

print(df_mail.head())

                                                메일제목  \
0  조아기프트(주) 이메일 발주서 입니다. (수령자:신현주) [ 2023-02-17 1...   
1  조아기프트(주) 이메일 발주서 입니다. (수령자:교사 곽승철(세종과학예술영재학교))...   
2  조아기프트(주) 이메일 발주서 입니다. (수령자:강연민 ) [ 2023-02-20 ...   
3  조아기프트(주) 이메일 발주서 입니다. (수령자:문창훈) [ 2023-02-28 0...   
4  조아기프트(주) 이메일 발주서 입니다. (수령자:홍서아) [ 2023-03-06 1...   

                                                  본문  \
0  발 주 서 ( 주문번호 : 5439674 )\n발 주 서\n(주)명성 귀하 (수령자...   
1  발 주 서 ( 주문번호 : 5439869 )\n발 주 서\n(주)명성 귀하 (수령자...   
2  발 주 서 ( 주문번호 : 5440210 )\n발 주 서\n(주)명성 귀하 (수령자...   
3  발 주 서 ( 주문번호 : 5442983 )\n샘플신청서\n(주)명성 귀하 (수령자...   
4  발 주 서 ( 주문번호 : 5444212 )\n발 주 서\n(주)명성 귀하 (수령자...   

                               메일키     UID                      메일폴더  \
0  &yHDFRK4w1QTSuA-_230217~|620810  620810  &yHDFRK4w1QTSuA-_230217~   
1  &yHDFRK4w1QTSuA-_230217~|620887  620887  &yHDFRK4w1QTSuA-_230217~   
2  &yHDFRK4w1QTSuA-_230217~|620899  620899  &yHDFRK4w1QTSuA-_230217~   
3  &yHDFRK4w1QTSuA-_230217~|621491  62

In [17]:
df_mail.to_excel(OUTPUT_DIR / "01-09_parsed_fields_draft.xlsx", index=False)

본문의 줄바꿈/공백을 `clean_text`로 정리한 뒤 저장합니다 → `01-10_mail_clean_final.xlsx` (`02_parse_and_merge`가 읽는 최종 입력)

In [18]:
df_mail['본문'] = df_mail['본문'].apply(clean_text)
df_mail.to_excel(OUTPUT_DIR / '01-10_mail_clean_final.xlsx', index=False)

정제된 본문(줄바꿈 제거)으로 `parse_order`를 다시 실행합니다. 정규식이 줄바꿈 없는 텍스트에서 더 안정적으로 매칭되기 때문에 재파싱합니다 → 결과는 다음 셀에서 `01-11_parsed_fields_qa.xlsx`로 저장(최종 QA용)

In [19]:
parsed_df = df_mail['본문'].apply(lambda x: parse_order(x) if isinstance(x, str) else parse_order("")).apply(pd.Series)

# clean_text 적용 후 재파싱한 결과로 덮어씀 (중복 컬럼 방지: pd.concat 대신 컬럼 재할당)
df_mail[parsed_df.columns] = parsed_df

print(df_mail.head())

                                                메일제목  \
0  조아기프트(주) 이메일 발주서 입니다. (수령자:신현주) [ 2023-02-17 1...   
1  조아기프트(주) 이메일 발주서 입니다. (수령자:교사 곽승철(세종과학예술영재학교))...   
2  조아기프트(주) 이메일 발주서 입니다. (수령자:강연민 ) [ 2023-02-20 ...   
3  조아기프트(주) 이메일 발주서 입니다. (수령자:문창훈) [ 2023-02-28 0...   
4  조아기프트(주) 이메일 발주서 입니다. (수령자:홍서아) [ 2023-03-06 1...   

                                                  본문  \
0  발 주 서 ( 주문번호 : 5439674 ) 발 주 서 (주)명성 귀하 (수령자:신...   
1  발 주 서 ( 주문번호 : 5439869 ) 발 주 서 (주)명성 귀하 (수령자:교...   
2  발 주 서 ( 주문번호 : 5440210 ) 발 주 서 (주)명성 귀하 (수령자:강...   
3  발 주 서 ( 주문번호 : 5442983 ) 샘플신청서 (주)명성 귀하 (수령자:문...   
4  발 주 서 ( 주문번호 : 5444212 ) 발 주 서 (주)명성 귀하 (수령자:홍...   

                               메일키     UID                      메일폴더  \
0  &yHDFRK4w1QTSuA-_230217~|620810  620810  &yHDFRK4w1QTSuA-_230217~   
1  &yHDFRK4w1QTSuA-_230217~|620887  620887  &yHDFRK4w1QTSuA-_230217~   
2  &yHDFRK4w1QTSuA-_230217~|620899  620899  &yHDFRK4w1QTSuA-_230217~   
3  &yHDFRK4w1QTSuA-_230217~|621491  62

In [20]:
df_mail.to_excel(OUTPUT_DIR / "01-11_parsed_fields_qa.xlsx", index=False)